In [ ]:
import sys
print("Executing Python from:", sys.executable)
try:
    import numpy
    print("NumPy successfully loaded from:", numpy.__file__)
except ImportError as e:
    print("NumPy import failed!")

print("\nCurrent sys.path:")
for path in sys.path:
    print(" -", path)

In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from makedf.makedf import *
from pyanalib.pandas_helpers import *
from makedf.util import *
from makedf.geniesyst import *
from tqdm import tqdm
from analysis_village.cc1pi.Constants import CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.dEdxCleaning import dEdxCleaning
import lmfit
import uproot as uproot
import os
import ROOT
import subprocess
from ROOT import std
import pickle
from itertools import combinations


# 1. Setup Paths
base_running_path = "."
base_path = base_running_path 

try:
    root_lib_dir = subprocess.check_output(['root-config', '--libdir'], text=True).strip()
except:
    root_lib_dir = "/cvmfs/larsoft.opensciencegrid.org/products/root/v6_28_12/Linux64bit+3.10-2.17-e26-p3915-prof/lib"

# 2. Update Environment
os.environ['LD_LIBRARY_PATH'] = f"{root_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
ROOT.gSystem.AddDynamicPath(root_lib_dir)

# 3. Add Include Path so classes can find each other's headers
ROOT.gInterpreter.AddIncludePath(base_path)

# 4. Explicit Compilation and Loading Function
def load_custom_class(class_name):
    source_file = os.path.join(base_path, f"{class_name}.cpp")
    header_file = os.path.join(base_path, f"{class_name}.h")
    so_file = os.path.join(base_path, f"{class_name}_cpp.so")
    
    # Compile with 'k' (keep), 'O' (optimize), 'f' (force)
    # Force is useful to ensure the symbol table is rebuilt correctly
    status = ROOT.gSystem.CompileMacro(source_file, "kOf")
    
    if status >= 0:
        # CRITICAL: Manually load the shared library to resolve symbols
        if ROOT.gSystem.Load(so_file) < 0:
            print(f"⚠️  Compiled {class_name} but failed to load {so_file}")
            return False
            
        # Declare the header to the interpreter to help dictionary lookup
        ROOT.gInterpreter.Declare(f'#include "{header_file}"')
        print(f"📦 Successfully compiled and linked: {class_name}")
        return True
    else:
        print(f"❌ Failed to compile: {class_name}")
        return False

# 5. Execute Sequence
# We must load PhysdEdx first as Hypfit depends on it
if load_custom_class("PhysdEdx"):
    if load_custom_class("Hypfit"):
        try:
            # Re-declaring just to be safe before instantiation
            ROOT.gInterpreter.ProcessLine(f'#include "{os.path.join(base_path, "Hypfit.h")}"')
            
            # Instantiate
            h_fit = ROOT.Hypfit()
            print("🚀 Success! Hypfit object initialized and linked.")
        except Exception as e:
            print(f"❌ Error during instantiation: {e}")
            # Final fallback: access via C++ global pointer if Python attribute fails
            ROOT.gInterpreter.ProcessLine("Hypfit* h_fit_ptr = new Hypfit();")
            h_fit = ROOT.h_fit_ptr
            print("🚀 Success! Hypfit object initialized via Global Pointer fallback.")

In [ ]:
#Load CV dataframe
keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_all.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

In [ ]:
rename_map = {
    ("pfp", "trk", "len", "", "", ""): "length",
    ("pfp", "trk", "rangeP", "p_muon", "", ""): "range_P",  # Change to 'p_pion' if analyzing pions
    ("pfp", "trk", "truth", "genp", "mag", ""): "true_P",
    ("pfp", "trk", "truth", "pur", "", ""): "purity",
    ("pfp", "trk", "truth", "eff", "", ""): "completeness",
    ("pfp", "trk", "truth", "p", "pdg", ""): "PDG",
    ("pfp", "trk", "truth", "p", "end_process", ""): "end_process",
    ("pfp", "best_plane", "", "", "", ""): "best_plane",
}
if "pion" in bnb_path:
    rename_map = {
        ("pfp", "trk", "len", "", "", ""): "length",
        ("pfp", "trk", "rangeP", "p_pion", "", ""): "range_P",  # Change to 'p_pion' if analyzing pions
        ("pfp", "trk", "truth", "genp", "mag", ""): "true_P",
        ("pfp", "trk", "truth", "pur", "", ""): "purity",
        ("pfp", "trk", "truth", "eff", "", ""): "completeness",
        ("pfp", "trk", "truth", "p", "pdg", ""): "PDG",
        ("pfp", "trk", "truth", "p", "end_process", ""): "end_process",
        ("pfp", "best_plane", "", "", "", ""): "best_plane",
    }

# Direct column reassignment bypasses MultiIndex structural restrictions
mc_bnb_pfp_df.columns = [
    rename_map.get(col, col) for col in mc_bnb_pfp_df.columns
]

# List of renamed columns to keep
target_columns = [
    "length",
    "range_P",
    "true_P",
    "purity",
    "completeness",
    "PDG",
    "end_process",
    "best_plane",
]

# Filter the DataFrame to keep only these target columns
mc_bnb_pfp_df = mc_bnb_pfp_df[
    [c for c in target_columns if c in mc_bnb_pfp_df.columns]
]

hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]

In [ ]:
import numpy as np
import pandas as pd

PROCESS_MAP = {
    0: "primary",
    1: "CoupledTransportation",
    2: "FastScintillation",
    3: "Decay",
    4: "anti_neutronInelastic",
    5: "neutronInelastic",
    6: "anti_protonInelastic",
    7: "protonInelastic",
    8: "hadInelastic",
    9: "pipInelastic",
    10: "pimInelastic",
    11: "xipInelastic",
    12: "ximInelastic",
    13: "kaonpInelastic",
    14: "kaonmInelastic",
    15: "sigmapInelastic",
    16: "sigmamInelastic",
    17: "kaon0LInelastic",
    18: "kaon0SInelastic",
    19: "lambdaInelastic",
    20: "anti_lambdaInelastic",
    21: "He3Inelastic",
    22: "ionInelastic",
    23: "xi0Inelastic",
    24: "alphaInelastic",
    25: "tInelastic",
    26: "dInelastic",
    27: "anti_neutronElastic",
    28: "neutronElastic",
    29: "anti_protonElastic",
    30: "protonElastic",
    31: "hadElastic",
    32: "pipElastic",
    33: "pimElastic",
    34: "kaonpElastic",
    35: "kaonmElastic",
    36: "conv",
    37: "phot",
    38: "annihil",
    39: "nCapture",
    40: "nKiller",
    41: "muMinusCaptureAtRest",
    42: "muIoni",
    43: "eBrem",
    44: "CoulombScat",
    45: "hBertiniCaptureAtRest",
    46: "hFritiofCaptureAtRest",
    47: "photonNuclear",
    48: "muonNuclear",
    49: "electronNuclear",
    50: "positronNuclear",
    51: "compt",
    52: "eIoni",
    53: "muBrems",
    54: "hIoni",
    55: "muPairProd",
    56: "hPairProd",
    57: "LArVoxelReadoutScoringProcess",
    58: "ionIoni",
    59: "hBrems",
    60: "Transportation",
    61: "msc",
    62: "StepLimiter",
    63: "LegacyUNKNOWN",
    64: "RadioactiveDecayBase",
    1024: "UNKNOWN",
}


def get_process_name(code):
    # Unpack scalar if code is wrapped in a Series, array, or list
    if isinstance(code, (pd.Series, np.ndarray, list)):
        if len(code) == 0:
            return "UNKNOWN"
        code = code.iloc[0] if isinstance(code, pd.Series) else code[0]

    try:
        if pd.isna(code):
            return "UNKNOWN"
        return PROCESS_MAP.get(int(code), "UNKNOWN")
    except (ValueError, TypeError):
        return "UNKNOWN"

def to_std_vector_double(arr):
    return ROOT.std.vector('double')(np.ascontiguousarray(arr, dtype=np.float64))


def get_track_hits(particle_idx, plane, hit_dfs):
    """Return (rr, dedx, pitch) numpy arrays for one particle on one plane, sorted by rr,
    with non-finite entries dropped -- the pandas equivalent of rr_vec->at(plane) etc.
    Returns None if this particle has no hits on that plane."""
    hit_df = hit_dfs[plane]
    try:
        hits = hit_df.loc[particle_idx]
    except KeyError:
        return None
    if isinstance(hits, pd.Series):
        # a single hit row -- promote to a 1-row frame
        hits = hits.to_frame().T

    hits = hits.sort_values('rr')
    rr    = hits['rr'].to_numpy(dtype=np.float64)
    dedx  = hits['dedx'].to_numpy(dtype=np.float64)
    pitch = hits['pitch'].to_numpy(dtype=np.float64)

    mask = np.isfinite(rr) & np.isfinite(dedx) & np.isfinite(pitch)
    return rr[mask], dedx[mask], pitch[mask]


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display
from scipy.optimize import curve_fit


# ----------------------------------------------------------------------
# Style helpers -- matched to the look of the reference figure
# (white background, boxed axes, blue circular markers, dotted grid)
# ----------------------------------------------------------------------
PLOT_STYLE = {
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.linewidth": 1.0,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.color": "0.6",
    "font.size": 11,
    "axes.labelsize": 12,
    "legend.frameon": False,
}


def _style_axis(ax, xlabel, ylabel, title=None):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, fontsize=11)
    ax.tick_params(direction="in", top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)


def _safe_idx_str(idx):
    """Turn a particle index (possibly a tuple like (24, 19, 1, 0)) into a
    filesystem-friendly string, e.g. '24_19_1_0' instead of '(24, 19, 1, 0)'."""
    if isinstance(idx, tuple):
        return "_".join(str(x) for x in idx)
    return str(idx).strip().replace(" ", "_").replace("(", "").replace(")", "").replace(",", "_")


# ----------------------------------------------------------------------
# Exponential model: fexp = A + B * exp(-C * x)
#   ROOT: TF1("fexp","[0]+[1]*exp(-[2]*x)", 0, 500)
#   Limits: A in [0,1e6], B in [0,1e6], C in [0,1e3]
# ----------------------------------------------------------------------
def exp_model(x, A, B, C):
    return A + B * np.exp(-C * x)


def fit_exp_dedx(rr, dedx, p0=(1.0, 1.0, 0.1)):
    """Fit dE/dx(rr) = A + B*exp(-C*rr), mirroring the ROOT TF1/Fit('0Q') call."""
    rr = np.asarray(rr, dtype=float)
    dedx = np.asarray(dedx, dtype=float)
    if len(rr) < 3:
        return None
    bounds = ([0.0, 0.0, 0.0], [1e6, 1e6, 1e3])
    try:
        popt, _ = curve_fit(exp_model, rr, dedx, p0=p0, bounds=bounds, maxfev=20000)
    except Exception as e:
        print(f"Exponential fit failed: {e}")
        return None
    return popt


# ----------------------------------------------------------------------
# Python port of Hypfit::get_hits_to_ignore that also returns the
# intermediate exponential fit + residual statistics, so everything can
# be plotted. Logic mirrors the C++ function step by step.
# ----------------------------------------------------------------------
def compute_hits_to_ignore_with_fit(
    rr,
    dedx,
    mode="all",
    N_skip=5,
    dEdx_truncate_upper=20.0,
    dEdx_truncate_below=0.5,
):
    rr = np.asarray(rr, dtype=float)
    dedx = np.asarray(dedx, dtype=float)
    n = len(rr)

    empty = {
        "ignore_stage1": set(range(n)),
        "ignore_final": set(range(n)),
        "fit_params": None,
        "residuals": np.full(n, np.nan),
        "mean": np.nan,
        "stdev": np.nan,
    }

    if n <= 6:
        return empty

    if "truncate_harsh" in mode:
        dEdx_truncate_upper = 5.0
        dEdx_truncate_below = 0.5

    ignore = set()
    for i in range(n):
        if dedx[i] < dEdx_truncate_below or dedx[i] > dEdx_truncate_upper:
            ignore.add(i)
            continue
        if "include_2cm" not in mode and rr[i] < 2:
            ignore.add(i)
            continue
        if i < N_skip:
            ignore.add(i)
            continue
        if i >= n - N_skip:
            ignore.add(i)
            continue

    ignore_stage1 = set(ignore)

    if "skip_only" in mode:
        return {
            "ignore_stage1": ignore_stage1,
            "ignore_final": ignore_stage1,
            "fit_params": None,
            "residuals": np.full(n, np.nan),
            "mean": np.nan,
            "stdev": np.nan,
        }

    keep_mask = np.array([i not in ignore for i in range(n)])
    rr_fit, dedx_fit = rr[keep_mask], dedx[keep_mask]
    popt = fit_exp_dedx(rr_fit, dedx_fit)
    if popt is None:
        return {
            "ignore_stage1": ignore_stage1,
            "ignore_final": ignore_stage1,
            "fit_params": None,
            "residuals": np.full(n, np.nan),
            "mean": np.nan,
            "stdev": np.nan,
        }

    residuals = np.full(n, np.nan)
    lo, hi = 5, n - 5
    s, cont = 0.0, 0
    for i in range(max(lo, 0), max(hi, 0)):
        if i not in ignore:
            resid = dedx[i] - exp_model(rr[i], *popt)
            residuals[i] = resid
            s += resid
            cont += 1
    mean = s / cont if cont > 0 else np.nan

    sq_sum = 0.0
    for i in range(max(lo, 0), max(hi, 0)):
        if i not in ignore and not np.isnan(residuals[i]):
            sq_sum += (residuals[i] - mean) ** 2
    stdev = np.sqrt(sq_sum / cont) if cont > 0 else np.nan

    ignore_final = set(ignore)
    for i in range(n):
        if i not in ignore_final and not np.isnan(residuals[i]):
            if abs(residuals[i] - mean) > 1.5 * stdev:
                ignore_final.add(i)

    return {
        "ignore_stage1": ignore_stage1,
        "ignore_final": ignore_final,
        "fit_params": popt,
        "residuals": residuals,
        "mean": mean,
        "stdev": stdev,
    }


# ----------------------------------------------------------------------
# The three diagnostic subplots
# ----------------------------------------------------------------------
def plot_dedx_diagnostics(
    fig,
    rr,
    dedx,
    result,
    particle_idx=None,
    dEdx_truncate_below=0.5,
    dEdx_truncate_upper=20.0,
):
    rr = np.asarray(rr, dtype=float)
    dedx = np.asarray(dedx, dtype=float)
    n = len(rr)

    axes = fig.subplots(1, 3)
    ax1, ax2, ax3 = axes

    popt = result["fit_params"]
    ignore_stage1 = result["ignore_stage1"]
    ignore_final = result["ignore_final"]
    residuals = result["residuals"]
    mean, stdev = result["mean"], result["stdev"]

    # ---- Panel 1: raw dE/dx profile (before hits are ignored) + fexp fit
    # Drop clear outliers (dE/dx > 20 or < 0.5 MeV/cm) purely for display,
    # matching the truncation thresholds used in get_hits_to_ignore.
    display_mask = (dedx >= 0.5) & (dedx <= 20.0)
    ax1.scatter(rr[display_mask], dedx[display_mask], s=18, color="tab:blue",
                edgecolor="none", zorder=3)
    if popt is not None:
        rr_disp = rr[display_mask] if np.any(display_mask) else rr
        x_fit = np.linspace(max(rr_disp.min(), 0), rr_disp.max(), 300)
        ax1.plot(
            x_fit,
            exp_model(x_fit, *popt),
            color="tab:red",
            lw=1.8,
            ls="--",
            label=f"fexp: A={popt[0]:.2f}, B={popt[1]:.2f}, C={popt[2]:.3f}",
            zorder=4,
        )
        ax1.legend(loc="upper right", fontsize=11)
    _style_axis(ax1, "Residual Range [cm]", "dE/dx [MeV/cm]", "Raw dE/dx profile + exp fit")

    # ---- Panel 2: residuals vs rr, band of width 1.5*stdev around mean
    valid = ~np.isnan(residuals)
    if popt is not None and np.any(valid):
        rr_v = rr[valid]
        res_v = residuals[valid]
        inside = np.abs(res_v - mean) <= 1.5 * stdev
        ax2.scatter(rr_v[inside], res_v[inside], s=18, color="tab:blue",
                    edgecolor="none", zorder=3, label="inside band")
        ax2.scatter(rr_v[~inside], res_v[~inside], s=18, color="tab:red",
                    edgecolor="none", zorder=3, label="outside band (outlier)")
        ax2.axhline(0, color="red", lw=1.5, ls="--", zorder=2, label="fit baseline (0)")
        x0, x1 = rr_v.min(), rr_v.max()
        ax2.axhline(mean + 1.5 * stdev, color="gray", lw=1.0, ls="--", zorder=2)
        ax2.axhline(mean - 1.5 * stdev, color="gray", lw=1.0, ls="--", zorder=2)
        ax2.axhspan(mean - 1.5 * stdev, mean + 1.5 * stdev, color="tab:blue", alpha=0.12, zorder=1)
        ax2.legend(loc="upper right", fontsize=11)
    else:
        ax2.text(0.5, 0.5, "No residuals available", ha="center", va="center",
                  transform=ax2.transAxes)
    _style_axis(ax2, "Residual Range [cm]", "dE/dx - fexp(rr) [MeV/cm]", "Residuals vs. exp fit")

    # ---- Panel 3: only the points NOT ignored -- excludes both the stage-1
    # (truncation/skip/2cm) cuts and the stage-2 (fit-residual outlier) cuts.
    excluded_all = ignore_stage1 | ignore_final
    good_mask = np.array([i not in excluded_all for i in range(n)])
    ax3.scatter(rr[good_mask], dedx[good_mask], s=18, color="tab:blue",
                edgecolor="none", zorder=3, label="kept hits")
    _style_axis(ax3, "Residual Range [cm]", "dE/dx [MeV/cm]", "Cleaned hits (outliers removed)")

    n_removed = len(ignore_final)
    fig.suptitle(
        f"Particle idx {particle_idx}  |  hits: {n}  |  removed: {n_removed}  |  kept: {n - n_removed}",
        fontsize=11,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.94])


# ----------------------------------------------------------------------
# Main widget
# ----------------------------------------------------------------------
def explore_pion_errs_widget(
    pfp_df,
    hit_dfs,
    h_fit,
    pdg=211,
    N_skip=5,
    hits_mode="all",
    save_dir="event_displays",
):
    """Interactive Jupyter Event Display using ipywidgets and C++ Hypfit bindings.

    Adds 3 side-by-side diagnostic subplots per event:
      1) Raw dE/dx vs. residual range (before any hit removal) + exponential fit
         fexp(rr) = A + B*exp(-C*rr)
      2) Residuals (dEdx - fexp(rr)) vs residual range, with a red baseline at 0
         and a shaded band of +/- 1.5*stdev around the residual mean. Points
         inside the band are blue, points outside (outliers) are red.
      3) The cleaned dE/dx profile -- only hits that survive
         Hypfit::get_hits_to_ignore (i.e. NOT flagged for removal).

    Includes a "Save Plot" button that writes the current figure to disk.
    """
    plt.rcParams.update(PLOT_STYLE)

    # 1. Fetch convolution and langau function maps for all planes upfront
    conv_map_og = h_fit.get_conv_function_map(pdg, "none", 1000)
    conv_map_shifted = h_fit.get_conv_function_map(pdg, "shift", 1000)
    langau_map = h_fit.get_langau_map(pdg, 500)

    valid_inelastic = {
        9, 10, 9.0, 10.0,
        "pipInelastic", "pimInelastic", "pi+Inelastic", "pi-Inelastic",
    }

    # 2. Pre-filter candidate event indices
    candidate_indices = []
    print("Pre-scanning dataset for matching pion inelastic events...")

    for particle_idx, pfp_row in pfp_df.iterrows():
        true_P = pfp_row.get("true_P", np.nan)
        if pd.isna(true_P):
            continue

        p_gev = true_P / 1000.0 if true_P > 20.0 else true_P
        if p_gev > 0.5:
            continue

        purity = pfp_row.get("purity", 1.0)
        completeness = pfp_row.get("completeness", 1.0)
        purity = 1.0 if pd.isna(purity) else purity
        completeness = 1.0 if pd.isna(completeness) else completeness
        if purity < 0.8 or completeness < 0.8:
            continue

        raw_end_proc = pfp_row.get("end_process", np.nan)
        if isinstance(raw_end_proc, (pd.Series, np.ndarray, list)):
            raw_end_proc = (
                raw_end_proc.iloc[0] if isinstance(raw_end_proc, pd.Series) else raw_end_proc[0]
            )

        end_proc_str = get_process_name(raw_end_proc)
        '''
        if raw_end_proc not in valid_inelastic and end_proc_str not in valid_inelastic:
            continue
        '''
        
        candidate_indices.append(particle_idx)

    print(f"Found {len(candidate_indices)} matching events out of {len(pfp_df)} particles.")
    if not candidate_indices:
        print("No events passed selection criteria. Please inspect DataFrame values.")
        return

    # 3. Create Widget UI Elements
    out = widgets.Output()
    prev_button = widgets.Button(description="◀ Previous", button_style="info", icon="arrow-left")
    next_button = widgets.Button(description="Next ▶", button_style="primary", icon="arrow-right")
    save_button = widgets.Button(description="💾 Save Plot", button_style="success", icon="save")
    save_status = widgets.Label(value="")
    slider = widgets.IntSlider(
        value=0, min=0, max=len(candidate_indices) - 1, step=1,
        description="Event:", continuous_update=False,
    )

    # Keep a reference to the currently drawn figure for the save button
    state = {"fig": None, "particle_idx": None}

    # 4. Event Rendering Callback
    def render_event(change_or_index):
        idx_pos = slider.value
        particle_idx = candidate_indices[idx_pos]
        pfp_row = pfp_df.loc[particle_idx]

        if isinstance(pfp_row, pd.DataFrame):
            pfp_row = pfp_row.iloc[0]

        true_P = pfp_row.get("true_P", np.nan)
        p_gev = true_P / 1000.0 if true_P > 20.0 else true_P

        best_plane_val = pfp_row.get("best_plane", 2)
        best_plane = int(best_plane_val) if pd.notna(best_plane_val) and best_plane_val >= 0 else 2
        used_plane = best_plane if (0 <= best_plane < 3) else 2

        hits = get_track_hits(particle_idx, used_plane, hit_dfs)
        if hits is None:
            with out:
                clear_output(wait=True)
                print(f"No hits found for Particle Index {particle_idx} on plane {used_plane}")
            save_status.value = ""
            return

        rr, dedx, pitch = hits
        rr = np.asarray(rr, dtype=float)
        dedx = np.asarray(dedx, dtype=float)
        if len(rr) == 0:
            with out:
                clear_output(wait=True)
                print(f"Empty hit arrays for Particle Index {particle_idx}")
            save_status.value = ""
            return

        # C++-side bad-hit mask (kept for reference / cross-check)
        bad_hits_cpp = h_fit.get_hits_to_ignore(rr, dedx, hits_mode)
        good_mask_cpp = ~np.isin(np.arange(len(rr)), bad_hits_cpp)
        rr_clean = rr[good_mask_cpp]
        dedx_clean = dedx[good_mask_cpp]  # available if you want to cross-check vs. python port

        # Python port used for the diagnostic plots (fit params + residuals)
        result = compute_hits_to_ignore_with_fit(rr, dedx, mode=hits_mode, N_skip=N_skip)

        with out:
            clear_output(wait=True)
            fig = plt.figure(figsize=(18, 5))
            plot_dedx_diagnostics(fig, rr, dedx, result, particle_idx=particle_idx)
            plt.show()
            state["fig"] = fig
            state["particle_idx"] = particle_idx
        save_status.value = ""

    # 5. Connect UI Controllers
    def on_prev_clicked(b):
        if slider.value > 0:
            slider.value -= 1

    def on_next_clicked(b):
        if slider.value < slider.max:
            slider.value += 1

    def on_save_clicked(b):
        import os
        if state["fig"] is None:
            save_status.value = "Nothing to save yet."
            return
        os.makedirs(save_dir, exist_ok=True)
        fname = os.path.join(save_dir, f"event_{_safe_idx_str(state['particle_idx'])}.png")
        state["fig"].savefig(fname, dpi=300, bbox_inches="tight")
        save_status.value = f"Saved to {fname}"

    prev_button.on_click(on_prev_clicked)
    next_button.on_click(on_next_clicked)
    save_button.on_click(on_save_clicked)
    slider.observe(render_event, names="value")

    # Display control layout and initial plot
    controls = widgets.HBox([prev_button, slider, next_button, save_button, save_status])
    display(widgets.VBox([controls, out]))
    render_event(None)

In [ ]:
# Usage execution:

# Execute Interactive Display Widget
explore_pion_errs_widget(
    mc_bnb_pfp_df,
    hit_dfs,
    h_fit,
    pdg=211,
    save_dir="/exp/sbnd/data/users/lpelegri/TLEGraphs/dEdxCleaning",
)



In [ ]:
print("DataFrame Columns:", mc_bnb_pfp_df.columns.tolist())
print("\nSample Selection Values:")
sample_cols = [
    c
    for c in ["true_P", "purity", "completeness", "end_process"]
    if c in mc_bnb_pfp_df.columns
]
print(mc_bnb_pfp_df[sample_cols].head(10))
print("\nUnique end_process values:", mc_bnb_pfp_df["end_process"].unique()[:10])